# 第14章 仮想空間に閉じた世界モデルを作る (Mesa の街)

第13章では実機のロボットの記録から世界モデルを作りました。実機の記録は集めるのに手間がかかり、
「もし別の動かし方をしたら」の答え合わせもできません。
この章では、**描画器 (ゲームのような仮想空間) が手元にある**場合を扱います。仮想空間なら、

- データはいくらでも、その場で作れる
- 想像した未来と、**本物の未来** (同じ行動を描画器で実行した結果) を並べて答え合わせができる
- 計画した行動を、そのまま本物の環境で実行して、ゴールに着いたかを確かめられる

という利点があります。題材は、街のシミュレーション Mesa の住民の目線です。目線の画像と「前進 / 左 / 右」の行動から、**次に何が見えるか**を予測する小さな世界モデルを 0 から学習します。
形は第13章と同じ DINO-WM 型です。

```
目線の画像 ──DINOv2 (凍結)──> 特徴 z_t ─┐
                                        ├─ 世界モデル (Transformer) ─> 次の特徴 z_{t+1}  ← 順動力学 = シミュレータ
行動 a_t (前進/左/右) ──────────────────┘
(z_t, z_{t+1}) ─ 逆動力学 ─> a_t                                                       ← 映像 → 行動
ゴールの画像の特徴 ─ 想像の中で行動を試す (CEM) ─> 次の一手                            ← 計画
```

データは、地図の生成器と、[mesa-clouds-renderer](https://github.com/oggata/mesa-clouds-renderer) の 3D 表示に寄せた
テクスチャ付きレイキャスタで、その場で作ります。建物は renderer と同じ写真 (`textures/v4`) と業種ごとの高さ (2x2 の大きな建物も)、
地面はアスファルト・歩道と路面標示・芝生・土 (`textures/base`, `textures/road/road_atlas.png`)、それに木と昼の空を描きます。

- ランタイム: **A100** か **L4** (全部で A100 なら 20〜30 分ほど、L4 はその 2〜3 倍の見込み)
- 学習に使う地図と、評価に使う地図は分けてあります (評価は**見たことのない街**で行います)
- Cosmos 3 のモデルは使いません (Hugging Face の利用申請も不要です)

In [ ]:
!apt-get -qq install -y fonts-noto-cjk > /dev/null
import os, math, random, time, json, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

# 日本語の見出しを出すためのフォント (Colab は apt で入れた Noto、Mac はヒラギノ)
import matplotlib.font_manager as fm
for f in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"):
    fm.fontManager.addfont(f)
_have = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = [f for f in ("Noto Sans CJK JP", "Hiragino Sans") if f in _have] + ["DejaVu Sans"]

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
QUICK = os.environ.get("WM_QUICK") == "1"   # 手元の動作確認用。Colab では False のまま
AMP = dict(device_type="cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda")
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
GRID = 30                    # 地図の大きさ (persona の学習と同じ)
IMG = 224                    # 描画の大きさ (DINOv2 の入力)
THUMB = 64                   # 目で見る用の縮小画像
TOK_GRID = 8                 # 特徴を 8×8 トークンに粗くする (16 = 粗くしない)
N_TOK = TOK_GRID ** 2
MOVE = 0.2                   # 1 歩で進む距離 (マス)
ROT = math.radians(12)       # 1 歩で曲がる角度
ACTIONS = ["前進", "左", "右", "後退"]   # 0〜2 は既存方策と同じ (0=forward 1=turn_left 2=turn_right)。3=後退はこの世界モデル用に足したもの
NA = len(ACTIONS)
HIST = 3                     # 世界モデルが見る過去のフレーム数

N_MAPS_TRAIN, N_MAPS_TEST = (4, 2) if QUICK else (64, 16)
EP_PER_MAP = 2 if QUICK else 6
EP_LEN = 20 if QUICK else 100      # 1 エピソードの歩数
WM_STEPS = 60 if QUICK else 12000
DEC_STEPS = 60 if QUICK else 3000

## 共通の部品 (DINOv2・世界モデル・復元器・逆動力学)

In [ ]:
# 凍結した DINOv2 (vits14)。224×224 の画像 → 16×16 のパッチ特徴 (各 384 次元)。
# 学習はしない。世界モデルはこの特徴の空間で「次に何が見えるか」を予測する。
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(DEVICE).eval()
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
FEAT_DIM = 384

@torch.no_grad()
def dino_encode(imgs, grid):
    """imgs: (B,3,224,224) float [0,1] → (B, grid*grid, 384)。grid<16 なら平均プーリングで粗くする"""
    with torch.autocast(**AMP):
        p = dino.forward_features((imgs - IMNET_MEAN) / IMNET_STD)["x_norm_patchtokens"]
    B = p.shape[0]
    p = p.float().transpose(1, 2).reshape(B, FEAT_DIM, 16, 16)
    if grid != 16:
        p = F.adaptive_avg_pool2d(p, grid)
    return p.flatten(2).transpose(1, 2)

In [ ]:
class WorldModel(nn.Module):
    """順動力学: 「過去 T フレームの特徴 + 各フレームでの行動」→「それぞれの次のフレームの特徴」。

    1 フレーム = N 個のパッチトークン。行動の埋め込みを全トークンに足し、フレーム単位の因果マスクで
    「フレーム i はフレーム 0..i だけを見る」ようにする (DINO-WM と同じ形)。
    出力は「前のフレームからの差分」。最初は差分 0 = 「何も変わらない」と予測する状態から学習が始まる。
    """
    def __init__(self, feat_dim, n_tok, hist, act_dim=None, n_actions=None, d=256, depth=6, heads=8):
        super().__init__()
        self.hist, self.n_tok = hist, n_tok
        self.inp = nn.Linear(feat_dim, d)
        if n_actions is not None:
            self.act = nn.Embedding(n_actions, d)
        else:
            self.act = nn.Sequential(nn.Linear(act_dim, d), nn.GELU(), nn.Linear(d, d))
        self.pos_tok = nn.Parameter(torch.randn(1, 1, n_tok, d) * 0.02)
        self.pos_t = nn.Parameter(torch.randn(1, hist, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.out = nn.Linear(d, feat_dim)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)
        f = torch.arange(hist).repeat_interleave(n_tok)
        self.register_buffer("mask", f[None, :] > f[:, None], persistent=False)  # True = 見てはいけない

    def forward(self, z, a):
        # z: (B,T,N,D) / a: (B,T) 行動番号 か (B,T,A) 連続値。位置 t の出力が t+1 の予測
        B, T, N, D = z.shape
        h = self.inp(z) + self.pos_tok + self.pos_t[:, -T:] + self.act(a)[:, :, None, :]
        h = self.blocks(h.reshape(B, T * N, -1), mask=self.mask[:T * N, :T * N])
        return z + self.out(self.norm(h)).reshape(B, T, N, D)


@torch.no_grad()
def imagine(model, z_ctx, a_seq):
    """想像の中で先へ進める (シミュレータとしての使い方)。
    z_ctx: (B,C,N,D) 実際に見た直近 C フレーム / a_seq: (B, C-1+K, ...) 文脈の先頭フレームからの行動
    返り値: (B,K,N,D) 想像した K フレーム先までの特徴。予測を次の入力に使い回すので、先ほど誤差が積もる"""
    H = model.hist
    zs = list(z_ctx.float().unbind(1)); C = len(zs)
    K = a_seq.shape[1] - (C - 1)
    for _ in range(K):
        t = len(zs) - 1; lo = max(0, t - H + 1)
        with torch.autocast(**AMP):
            pred = model(torch.stack(zs[lo:], 1), a_seq[:, lo:t + 1])[:, -1]
        zs.append(pred.float())
    return torch.stack(zs[C:], 1)


class Decoder(nn.Module):
    """特徴 (grid×grid トークン) → 画像 (3, 8·grid, 8·grid)。想像した未来を目で見るためだけに使う。
    世界モデルの学習や計画には一切使わない (画素ではなく特徴で考えるのが DINO-WM の要点)。"""
    def __init__(self, feat_dim, grid, ch=128):
        super().__init__()
        self.grid = grid
        def up(i, o):
            return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.GroupNorm(8, o), nn.GELU(),
                                 nn.Conv2d(o, o, 3, 1, 1), nn.GroupNorm(8, o), nn.GELU())
        self.net = nn.Sequential(nn.Conv2d(feat_dim, 2 * ch, 1), up(2 * ch, 2 * ch), up(2 * ch, ch),
                                 up(ch, ch // 2), nn.Conv2d(ch // 2, 3, 3, 1, 1))

    def forward(self, z):
        B, N, D = z.shape
        return torch.sigmoid(self.net(z.transpose(1, 2).reshape(B, D, self.grid, self.grid)))


class InverseDynamics(nn.Module):
    """逆動力学: (今のフレーム, 次のフレーム) の特徴 → その間の行動。「映像 → 行動 (座標)」の部分"""
    def __init__(self, feat_dim, n_tok, out_dim, r=32):
        super().__init__()
        self.red = nn.Linear(feat_dim, r)
        self.mlp = nn.Sequential(nn.Linear(2 * n_tok * r, 512), nn.GELU(),
                                 nn.Linear(512, 512), nn.GELU(), nn.Linear(512, out_dim))

    def forward(self, z0, z1):
        return self.mlp(torch.cat([self.red(z0).flatten(1), self.red(z1).flatten(1)], 1))


def valid_starts(ep_start, ep_len, H):
    """窓 (最大 H+1 フレーム) が 1 つのエピソードに収まる開始位置の一覧"""
    return torch.cat([torch.arange(s, s + n - H) for s, n in zip(ep_start, ep_len) if n > H])


def train_wm(model, feats, acts, starts, steps, batch=64, lr=3e-4, log_every=500):
    """feats: (M,N,D) 全フレームを並べた特徴 / acts: (M,) か (M,A) フレーム i で取った行動 / starts: valid_starts"""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.05)
    H = model.hist; losses = []; t0 = time.time()
    model.train()
    for step in range(steps):
        T = random.randint(1, H)                       # 文脈の長さもばらす (始めのうちは過去が短いので)
        i = starts[torch.randint(0, len(starts), (batch,))]
        idx = (i[:, None] + torch.arange(T + 1)).to(feats.device)
        z = feats[idx].float()                          # (B,T+1,N,D)
        a = acts[idx[:, :T]]
        with torch.autocast(**AMP):
            pred = model(z[:, :T], a)
        loss = F.mse_loss(pred.float(), z[:, 1:])
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); losses.append(loss.item())
        if (step + 1) % log_every == 0 or step + 1 == steps:
            print(f"step {step+1:6d}  loss {np.mean(losses[-log_every:]):.4f}  ({time.time()-t0:.0f}s)")
    model.eval()
    return losses


def train_decoder(dec, feats, thumbs, steps, batch=64, lr=1e-3, log_every=500):
    """thumbs: (M,H,W,3) uint8。特徴 → 画像 を学習する"""
    opt = torch.optim.AdamW(dec.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    dec.train(); losses = []
    for step in range(steps):
        i = torch.randint(0, len(feats), (batch,), device=feats.device)
        x = thumbs[i].permute(0, 3, 1, 2).float() / 255
        with torch.autocast(**AMP):
            y = dec(feats[i].float())
        loss = F.l1_loss(y.float(), x)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        losses.append(loss.item())
        if (step + 1) % log_every == 0 or step + 1 == steps:
            print(f"decoder step {step+1:6d}  L1 {np.mean(losses[-log_every:]):.4f}")
    dec.eval()


@torch.no_grad()
def decode(dec, z):
    """(…,N,D) → (…,H,W,3) numpy [0,1]"""
    sh = z.shape[:-2]
    with torch.autocast(**AMP):
        img = dec(z.reshape(-1, *z.shape[-2:]).float()).float()
    return img.permute(0, 2, 3, 1).reshape(*sh, *img.shape[2:], 3).cpu().numpy()

In [ ]:
def show_rows(rows, cols_title=None, size=1.6, title=None):
    """rows: {行の名前: [画像 (H,W,3), ...]} を表にして並べる"""
    names = list(rows); n = max(len(v) for v in rows.values())
    fig, ax = plt.subplots(len(names), n, figsize=(size * n * 1.2, size * len(names)), squeeze=False)
    for r, name in enumerate(names):
        for c in range(n):
            a = ax[r, c]; a.axis("off")
            if c < len(rows[name]):
                a.imshow(np.clip(rows[name][c], 0, 1))
            if c == 0:
                a.text(-0.08, 0.5, name, transform=a.transAxes, ha="right", va="center", fontsize=10)
            if r == 0 and cols_title is not None:
                a.set_title(cols_title[c], fontsize=9)
    if title: fig.suptitle(title)
    plt.tight_layout(); plt.show()


def animate_rows(rows, fps=6, size=2.2):
    """rows: {名前: (T,H,W,3)} を横に並べた動画 (Colab の中で再生できる HTML)"""
    names = list(rows); T = min(len(v) for v in rows.values())
    fig, ax = plt.subplots(1, len(names), figsize=(size * len(names) * 1.2, size + 0.4), squeeze=False)
    ims = []
    for a, name in zip(ax[0], names):
        a.axis("off"); a.set_title(name, fontsize=10)
        ims.append(a.imshow(np.clip(rows[name][0], 0, 1)))
    def upd(t):
        for im, name in zip(ims, names):
            im.set_data(np.clip(rows[name][t], 0, 1))
        return ims
    anim = animation.FuncAnimation(fig, upd, frames=T, interval=1000 / fps)
    plt.close(fig)
    return HTML(anim.to_jshtml())

## データを作る (地図 → 目線の描画 → DINOv2 の特徴)

In [ ]:
# ── 地図: persona の学習ノートブックの make_map_organic と同じ作り方 ──
OTHER, ROAD, BUILDING, TREE = 0, 1, 2, 3
from collections import deque

def make_map_organic(size, seed):
    rng = random.Random(seed)
    grid = np.full((size, size), OTHER, dtype=np.int32)
    rows = cols = list(range(0, size, 4))
    for r in rows: grid[r, :] = ROAD
    for c in cols: grid[:, c] = ROAD
    for ri in range(len(rows) - 1):
        for ci in range(len(cols) - 1):
            cells = [(r, c) for r in range(rows[ri] + 1, rows[ri + 1]) for c in range(cols[ci] + 1, cols[ci + 1])]
            if not cells: continue
            b = rng.choice(cells); grid[b] = BUILDING
            for rc in cells:
                if rc == b: continue
                v = rng.random()
                grid[rc] = TREE if v < 0.25 else BUILDING if v < 0.45 else grid[rc]
    for r in rows: grid[r, :] = ROAD
    for c in cols: grid[:, c] = ROAD

    def connected(g):
        cells = list(zip(*np.where(g == ROAD)))
        seen = {cells[0]}; q = deque([cells[0]])
        while q:
            r, c = q.popleft()
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                n = (r + dr, c + dc)
                if 0 <= n[0] < size and 0 <= n[1] < size and n not in seen and g[n] == ROAD:
                    seen.add(n); q.append(n)
        return len(seen) == len(cells)

    cand = [(r, c) for r, c in zip(*np.where(grid == ROAD)) if not (r in rows and c in cols)]
    rng.shuffle(cand)
    quota = int(len(cand) * (0.25 + rng.random() * 0.25)); removed = 0
    for rc in cand:
        if removed >= quota: break
        grid[rc] = OTHER
        if connected(grid):
            grid[rc] = TREE if rng.random() < 0.4 else OTHER; removed += 1
        else:
            grid[rc] = ROAD
    return grid


# ── 建物の種類と高さ: mesa-clouds-renderer の BLDG_TYPES (server.js) と同じ ──
#   (名前 = textures/v4/<名前>.jpg, 敷地 1=1x1 / 2=2x2 マス, 高さ [マス])
BLDG_TYPES = [
    ("kiosk", 1, 0.7), ("conbini", 1, 0.9), ("pharmacy", 1, 0.9), ("cafe", 1, 1.1), ("gyudon", 1, 1.1),
    ("ramen", 1, 1.1), ("bento", 1, 1.1), ("shop", 1, 1.4), ("house", 1, 1.4), ("post", 1, 1.4),
    ("bank", 1, 1.7), ("apartment", 1, 2.1), ("hotel", 1, 2.1), ("office", 1, 2.6), ("tower", 1, 3.3),
    ("police", 1, 1.7), ("elementary", 1, 1.1), ("junior", 1, 1.4), ("warehouse", 1, 1.4),
    ("supermarket", 2, 1.1), ("temple", 2, 1.1), ("school", 2, 1.4), ("station", 2, 1.4), ("library", 2, 1.4),
    ("hospital", 2, 1.7), ("cityhall", 2, 1.7), ("museum", 2, 1.7), ("stadium", 2, 2.1), ("mall", 2, 2.6),
    ("high", 2, 1.7), ("university", 2, 2.1),
]
SMALL = [i for i, t in enumerate(BLDG_TYPES) if t[1] == 1]
BIG = [i for i, t in enumerate(BLDG_TYPES) if t[1] == 2]
P_BIG = 0.3          # 街区 (道に囲まれた 3x3) に 2x2 の大きな建物を置く割合

def bldg_height(t, r, c):
    """業種の高さ × 座標から決まる倍率 (±12% を 5 段階)。renderer と同じく、同じ業種が定規で揃わないように"""
    k = ((r * 73856093) ^ (c * 19349663)) % 5
    return BLDG_TYPES[t][2] * (0.88 + 0.06 * k)

def place_buildings(grid, seed):
    """建物のマスに業種を割り当てる。返り値: 地図 (2x2 を足したもの) と、マスごとの
    業種 bt (建物でなければ -1)・高さ bh・建物の左上のマス org・敷地の幅 fp"""
    rng = np.random.default_rng(seed)
    grid = grid.copy(); S = grid.shape[0]
    bt = np.full(grid.shape, -1, np.int64); bh = np.zeros(grid.shape, np.float32)
    org = np.zeros(grid.shape + (2,), np.int64); fp = np.ones(grid.shape, np.int64)
    # 1) いくつかの街区に 2x2 の大きな建物 (スーパー・学校・病院 …)。街区の中なので道は塞がない
    for r0 in range(1, S - 3, 4):
        for c0 in range(1, S - 3, 4):
            if rng.random() >= P_BIG: continue
            r, c = r0 + int(rng.integers(0, 2)), c0 + int(rng.integers(0, 2))
            t = int(rng.choice(BIG)); sl = (slice(r, r + 2), slice(c, c + 2))
            grid[sl] = BUILDING; bt[sl] = t; org[sl] = (r, c); fp[sl] = 2; bh[sl] = bldg_height(t, r, c)
    # 2) 残りの建物のマスは 1x1 の店や家
    for r, c in zip(*np.where((grid == BUILDING) & (bt < 0))):
        t = int(rng.choice(SMALL)); bt[r, c] = t; org[r, c] = (r, c); bh[r, c] = bldg_height(t, r, c)
    return grid, bt, bh, org, fp


# ── 地面: renderer の rebuildGround と同じ貼り分け ──
#   道 → アスファルト + 路面標示 (歩道・白線・横断歩道)、建物の足元 → 舗装、木の足元・空き地 → 芝生、
#   踏み固められた空き地 → 土 (renderer は人の通った回数で決める。ここでは座標から 3 割)
G_ASPHALT, G_GRASS, G_DIRT = 0, 1, 2
SLOT_TWOLANE, SLOT_ONEWAY, SLOT_PATH = 0, 16, 32   # 路面標示アトラスの枠 (roads.js と同じ)

def ground_layout(grid):
    S = grid.shape[0]
    gk = np.full(grid.shape, G_GRASS, np.int64); slot = np.full(grid.shape, -1, np.int64)
    for r in range(S):
        for c in range(S):
            t = grid[r, c]
            if t in (ROAD, BUILDING): gk[r, c] = G_ASPHALT
            elif t == OTHER and ((r * 92837111) ^ (c * 689287499)) % 10 < 3: gk[r, c] = G_DIRT
            if t != ROAD: continue
            R = lambda rr, cc: 0 <= rr < S and 0 <= cc < S and grid[rr, cc] == ROAD
            mask = R(r - 1, c) | R(r, c + 1) << 1 | R(r + 1, c) << 2 | R(r, c - 1) << 3   # N|E|S|W
            slot[r, c] = SLOT_PATH if bin(mask).count("1") <= 1 else SLOT_ONEWAY + mask
    return gk, slot


t0 = time.time()
MAP_SEEDS = list(range(N_MAPS_TRAIN)) + [10_000 + s for s in range(N_MAPS_TEST)]   # 後ろの N_MAPS_TEST 枚は評価専用
MAPS, _bt, _bh, _org, _fp, _gk, _slot = [], [], [], [], [], [], []
for s in MAP_SEEDS:
    g, bt, bh, org, fp = place_buildings(make_map_organic(GRID, s), s)
    gk, slot = ground_layout(g)
    for lst, v in zip((MAPS, _bt, _bh, _org, _fp, _gk, _slot), (g, bt, bh, org, fp, gk, slot)): lst.append(v)
_T = lambda a, **k: torch.tensor(np.stack(a), device=DEVICE, **k)
BT, BH, BORG, BFP = _T(_bt), _T(_bh, dtype=torch.float32), _T(_org), _T(_fp)   # BT[地図, x, y] = 業種 (建物でなければ -1)
GK, RSLOT = _T(_gk), _T(_slot)
TREEM = _T([g == TREE for g in MAPS])                                        # 木のマス (通れるが、目には見える)
print(f"地図 {len(MAPS)} 枚 ({time.time()-t0:.0f}s)")

cmap = plt.matplotlib.colors.ListedColormap(["#1a3a6a", "#d0d0c8", "#c42020", "#256b28"])
fig, ax = plt.subplots(2, 4, figsize=(12, 6))
for j, k in enumerate([0, 1, N_MAPS_TRAIN, N_MAPS_TRAIN + 1]):
    ax[0, j].imshow(MAPS[k], cmap=cmap, vmin=0, vmax=3)
    ax[0, j].set_title(("学習用" if k < N_MAPS_TRAIN else "評価用") + f" seed={MAP_SEEDS[k]}", fontsize=9)
    hcm = plt.get_cmap("viridis").copy(); hcm.set_bad("#e4e4e0")          # 建物でないマスは薄い灰色
    im = ax[1, j].imshow(np.ma.masked_where(_bt[k] < 0, _bh[k]), cmap=hcm, vmin=0.5, vmax=3.7)
    ax[1, j].set_title("建物の高さ [マス]", fontsize=9)
for a in ax.flat: a.axis("off")
fig.colorbar(im, ax=ax[1].tolist(), shrink=0.8)
plt.show()

In [ ]:
# ── 目線の描画: mesa-clouds-renderer の 3D 表示に寄せたテクスチャ付き DDA レイキャスタ ──
#   ・建物の壁 = renderer と同じ写真 (textures/v4)。1 枚で建物の正面まるごとの絵なので、壁の高さ全体に貼る
#   ・建物の高さ = 業種ごとの高さ。手前の低い建物の上から、奥の高いビルがのぞいて見える
#   ・地面 = アスファルト + 路面標示 (歩道・白線・横断歩道) / 芝生 / 土。renderer の textures/base と road_atlas
#   ・空 = renderer の昼の空 (地平線が明るく、上ほど少し濃い)
import urllib.request, io
from PIL import Image

RAW = "https://raw.githubusercontent.com/oggata/mesa-clouds-renderer/main/"
def fetch(path, mode="RGB"):
    return Image.open(io.BytesIO(urllib.request.urlopen(RAW + path).read())).convert(mode)
def to_t(im, size):
    return torch.tensor(np.asarray(im.resize(size, Image.BILINEAR), np.float32) / 255, device=DEVICE)

TW, TH = 192, 256           # 壁の写真 (縦長の写真なので縦を多めに。近づいてもぼやけない大きさ)
GT = 128                    # 地面の模様 1 マスぶん
AT = 64                     # 路面標示の 1 枠
TEX = torch.stack([to_t(fetch(f"textures/v4/{n}.jpg"), (TW, TH)) for n, _, _ in BLDG_TYPES])   # (業種,TH,TW,3)
_dirt = to_t(fetch("textures/base/other_256.jpg"), (GT, GT)) * torch.tensor([0xb5, 0xb0, 0x8a], device=DEVICE) / 255 * 1.55
_asph = to_t(fetch("textures/base/asphalt_256.jpg"), (GT, GT))
_asph = _asph / _asph.mean((0, 1)) * torch.tensor([0.265, 0.275, 0.295], device=DEVICE)   # 車道の色 (roads.js の FLOOR_RGB.ASPHALT)
GTEX = torch.stack([_asph,
                    to_t(fetch("textures/base/grass_256.jpg"), (GT, GT)), _dirt.clamp(0, 1)])  # G_ASPHALT, G_GRASS, G_DIRT
_atlas = fetch("textures/road/road_atlas.png", "RGBA")   # 8x8 枠 (1 枠 128px、まわり 8px は継ぎ目用の余白)
ATLAS = torch.stack([to_t(_atlas.crop(((s % 8) * 128 + 8, (s // 8) * 128 + 8, (s % 8) * 128 + 120, (s // 8) * 128 + 120)), (AT, AT))
                     for s in range(33)])               # (33,AT,AT,4)。車道の内側は透明 (下のアスファルトが見える)

def make_tree_sprite(n=64, seed=0):
    """木の絵 (RGBA)。幹 + 丸い樹冠。renderer の木と同じく、丸い緑の頭に茶色の幹"""
    g = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:n, 0:n] / n
    crown = ((xx - 0.5) / 0.48) ** 2 + ((yy - 0.36) / 0.36) ** 2 < 1
    trunk = (abs(xx - 0.5) < 0.06) & (yy > 0.55)
    blot = np.kron(g.standard_normal((n // 8, n // 8)), np.ones((8, 8)))      # 葉のまだら (幅の 1/8 のかたまり)
    shade = 0.75 + 0.35 * (0.5 - yy) + 0.10 * blot                             # 上が明るい
    rgb = np.where(crown[..., None], np.array([0x48, 0x90, 0x3f]) / 255 * shade[..., None],
                   np.array([0x7a, 0x52, 0x30]) / 255)
    a = (crown | trunk).astype(np.float32)
    return torch.tensor(np.dstack([rgb.clip(0, 1), a]), device=DEVICE, dtype=torch.float32)
TREE_TEX = make_tree_sprite(128)
TREE_H, TREE_R = 1.0, 0.32  # 木の高さと半径 [マス]

def _rgb(h): return torch.tensor([(h >> 16) & 255, (h >> 8) & 255, h & 255], device=DEVICE, dtype=torch.float32) / 255
SKY_H = _rgb(0xeaf2f7)                                    # 地平線 (renderer の昼の背景色 = フォグの色)
SKY_Z = SKY_H * 0.78 + (_rgb(0x1b3a63) - SKY_H * 0.78) * 0.10   # 天頂 (renderer の空ドームと同じ混ぜ方)
FOV = math.radians(68)      # 視野角 (renderer の一人称 FPV_FOV)
EYE = 0.22                  # 目の高さ [マス] (renderer の住民の目線 0.44 / CELL 2.0)
SSAA = 2                    # 2 倍の大きさで描いて縮める (地面の模様のちらつき止め)
LAYERS = 4                  # 1 列に重ねて描く壁の枚数 (手前の低い建物の奥にある高い建物を見せる)
TLAYERS = 3                 # 1 列に重ねて描く木の枚数
FOG0, FOG1 = 6.0, 30.0      # この距離 [マス] から空の色に溶けはじめ、FOG1 で 7 割

def _fog(col, dist):
    f = ((dist - FOG0) / (FOG1 - FOG0)).clamp(0, 1) * 0.7
    return col * (1 - f[..., None]) + SKY_H * f[..., None]

@torch.no_grad()
def _render(xs, ys, ths, mid, H, W, MAXSTEPS=64):
    """(N,) → (N,H,W,3)。列 0 が th - FOV/2 を見る = th を減らすと画面の左を向く (既存方策の turn_left と同じ)"""
    N = xs.shape[0]
    cam = 2 * torch.arange(W, device=DEVICE, dtype=torch.float32) / W - 1
    dirX, dirY = torch.cos(ths), torch.sin(ths)
    pl = math.tan(FOV / 2); planeX, planeY = -dirY * pl, dirX * pl
    FOC = (W / 2) / pl                      # 焦点距離 [px]。縦も横と同じ倍率で写す (縦に伸びない)
    hor = H / 2
    yc = torch.arange(H, device=DEVICE, dtype=torch.float32) + 0.5   # 画素の中心の行
    rdx = dirX[:, None] + planeX[:, None] * cam[None]; rdy = dirY[:, None] + planeY[:, None] * cam[None]
    m3 = mid[:, None, None]

    # ── 空 ──
    t = ((hor - yc) / hor).clamp(0, 1) ** 0.65
    img = (SKY_H + (SKY_Z - SKY_H) * t[:, None])[None, :, None, :].expand(N, H, W, 3).clone()

    # ── 地面 (床キャスト): 地平線より下の行は、その行が指す地面の 1 点の色 ──
    fr = yc > hor
    dist = FOC * EYE / (yc[fr] - hor)                                    # (Hf,) その行の地面までの距離
    fx = xs[:, None, None] + rdx[:, None, :] * dist[None, :, None]       # (N,Hf,W)
    fy = ys[:, None, None] + rdy[:, None, :] * dist[None, :, None]
    r, c = torch.floor(fx).long(), torch.floor(fy).long()
    inb = (r >= 0) & (r < GRID) & (c >= 0) & (c < GRID)
    rr, cc = r.clamp(0, GRID - 1), c.clamp(0, GRID - 1)
    fu, fv = fy - torch.floor(fy), fx - torch.floor(fx)                  # u = 列 (東) 方向, v = 行 (南) 方向
    base = GTEX[GK[m3, rr, cc], (fv * GT).long().clamp(0, GT - 1), (fu * GT).long().clamp(0, GT - 1)]
    slot = RSLOT[m3, rr, cc]
    mk = ATLAS[slot.clamp(min=0), (fv * AT).long().clamp(0, AT - 1), (fu * AT).long().clamp(0, AT - 1)]
    a = mk[..., 3:] * (slot >= 0)[..., None]
    floor = _fog(base * (1 - a) + mk[..., :3] * a, dist[None, :, None].expand_as(fx))
    img[:, fr] = torch.where(inb[..., None], floor, SKY_H)               # 地図の外は何も無い (renderer と同じく背景色)

    # ── 壁: DDA でレイを進め、「手前の壁の上端より上に見える壁」を最大 LAYERS 枚まで拾う ──
    mapX = torch.floor(xs).long()[:, None].expand(N, W).clone()
    mapY = torch.floor(ys).long()[:, None].expand(N, W).clone()
    ddx = torch.where(rdx == 0, torch.full_like(rdx, 1e30), (1 / rdx).abs())
    ddy = torch.where(rdy == 0, torch.full_like(rdy, 1e30), (1 / rdy).abs())
    stepX = torch.where(rdx < 0, -1, 1); stepY = torch.where(rdy < 0, -1, 1)
    pX, pY = xs[:, None], ys[:, None]
    sdx = torch.where(rdx < 0, (pX - mapX) * ddx, (mapX + 1 - pX) * ddx)
    sdy = torch.where(rdy < 0, (pY - mapY) * ddy, (mapY + 1 - pY) * ddy)
    side = torch.zeros((N, W), dtype=torch.long, device=DEVICE)
    active = torch.ones((N, W), dtype=torch.bool, device=DEVICE)
    occ = torch.full((N, W), float(H), device=DEVICE)        # この行より下は手前の壁で埋まっている
    cnt = torch.zeros((N, W), dtype=torch.long, device=DEVICE)
    L = {k: torch.zeros((LAYERS, N, W), device=DEVICE) for k in ("perp", "top", "bot", "u", "occ", "side")}
    Lt = torch.full((LAYERS, N, W), -1, dtype=torch.long, device=DEVICE)
    tcnt = torch.zeros((N, W), dtype=torch.long, device=DEVICE)
    TLr = {k: torch.zeros((TLAYERS, N, W), device=DEVICE) for k in ("dz", "u", "h", "occ")}
    TLv = torch.zeros((TLAYERS, N, W), dtype=torch.bool, device=DEVICE)
    m = mid[:, None].expand(N, W)
    for _ in range(MAXSTEPS):
        xlt = sdx < sdy; sx = active & xlt; sy = active & ~xlt
        sdx = torch.where(sx, sdx + ddx, sdx); mapX = torch.where(sx, mapX + stepX, mapX); side = torch.where(sx, 0, side)
        sdy = torch.where(sy, sdy + ddy, sdy); mapY = torch.where(sy, mapY + stepY, mapY); side = torch.where(sy, 1, side)
        inb = (mapX >= 0) & (mapX < GRID) & (mapY >= 0) & (mapY < GRID)
        mx, my = mapX.clamp(0, GRID - 1), mapY.clamp(0, GRID - 1)
        t = BT[m, mx, my]
        perp = torch.where(side == 0, sdx - ddx, sdy - ddy).clamp(min=1e-4)
        top = hor - FOC * (BH[m, mx, my] - EYE) / perp
        vis = active & inb & (t >= 0) & (top < occ) & (cnt < LAYERS)
        # 壁の横方向の位置 u (0..1)。2x2 の建物は 2 マスにまたがって 1 枚の写真を貼る
        o, fpw = BORG[m, mx, my], BFP[m, mx, my]
        u = torch.where(side == 0, pY + perp * rdy - o[..., 1], pX + perp * rdx - o[..., 0]) / fpw
        flip = ((side == 0) & (rdx > 0)) | ((side == 1) & (rdy < 0)); u = torch.where(flip, 1 - u, u)
        # 木: マスの中心に立つ、こちらを向いた板。通り抜けられるので壁と違って奥を隠さない (絵の透明部分は透ける)
        dz = (mx + 0.5 - pX) * dirX[:, None] + (my + 0.5 - pY) * dirY[:, None]            # 前方への距離
        s_ = (pX + rdx * dz - mx - 0.5) * -dirY[:, None] + (pY + rdy * dz - my - 0.5) * dirX[:, None]  # 横のずれ
        th_ = TREE_H * (0.85 + 0.075 * (((mx * 73856093) ^ (my * 19349663)) % 5))
        tvis = (active & inb & TREEM[m, mx, my] & (dz > 0.3) & (s_.abs() < TREE_R) & (tcnt < TLAYERS)
                & (hor - FOC * (th_ - EYE) / dz.clamp(min=1e-4) < occ))
        for k in range(TLAYERS):
            sel = tvis & (tcnt == k)
            TLv[k] |= sel
            for name, v in (("dz", dz), ("u", s_ / (2 * TREE_R) + 0.5), ("h", th_.float()), ("occ", occ)):
                TLr[name][k] = torch.where(sel, v, TLr[name][k])
        tcnt = tcnt + tvis.long()
        for k in range(LAYERS):
            sel = vis & (cnt == k)
            Lt[k] = torch.where(sel, t, Lt[k])
            for name, v in (("perp", perp), ("top", top), ("bot", hor + FOC * EYE / perp), ("u", u),
                            ("occ", occ), ("side", side.float())):
                L[name][k] = torch.where(sel, v, L[name][k])
        occ = torch.where(vis, top, occ); cnt = cnt + vis.long()
        active = active & inb & (occ > 0) & (cnt < LAYERS)
        if not active.any(): break

    rows = yc[None, :, None]
    for k in range(LAYERS):
        tk = Lt[k]
        top, bot = L["top"][k][:, None, :], L["bot"][k][:, None, :]
        inwall = (tk >= 0)[:, None, :] & (rows >= top) & (rows < bot) & (rows < L["occ"][k][:, None, :])
        ty = ((rows - top) / (bot - top)).clamp(0, 0.999) * TH          # 写真は壁の高さ全体に貼る
        tx = (L["u"][k].clamp(0, 0.999) * TW).long()
        col = TEX[tk.clamp(min=0)[:, None, :].expand(N, H, W), ty.long(), tx[:, None, :].expand(N, H, W)]
        shade = torch.where(L["side"][k] > 0, 0.82, 1.0)                   # 面の向きで明るさを変える (日なた / 日かげ)
        col = _fog(col * shade[:, None, :, None], L["perp"][k][:, None, :].expand(N, H, W))
        img = torch.where(inwall[..., None], col, img)
    for k in reversed(range(TLAYERS)):                                     # 木は奥から順に重ねる
        dz = TLr["dz"][k].clamp(min=1e-4)[:, None, :]
        top = hor - FOC * (TLr["h"][k][:, None, :] - EYE) / dz; bot = hor + FOC * EYE / dz
        ty = ((rows - top) / (bot - top)).clamp(0, 0.999) * TREE_TEX.shape[0]
        tx = (TLr["u"][k].clamp(0, 0.999) * TREE_TEX.shape[1]).long()[:, None, :].expand(N, H, W)
        tc = TREE_TEX[ty.long().expand(N, H, W), tx]
        on = TLv[k][:, None, :] & (rows >= top) & (rows < bot) & (rows < TLr["occ"][k][:, None, :]) & (tc[..., 3] > 0.5)
        img = torch.where(on[..., None], _fog(tc[..., :3], dz.expand(N, H, W)), img)
    return img

@torch.no_grad()
def render(xs, ys, ths, mid, H=IMG, W=IMG, chunk=32):
    """xs, ys, ths, mid (地図の番号): (N,) → 目線の画像 (N,3,H,W) [0,1]"""
    out = []
    for i in range(0, xs.shape[0], chunk):
        sl = slice(i, i + chunk)
        im = _render(xs[sl], ys[sl], ths[sl], mid[sl], H * SSAA, W * SSAA).permute(0, 3, 1, 2)
        out.append(F.avg_pool2d(im, SSAA) if SSAA > 1 else im)
    return torch.cat(out).clamp(0, 1).contiguous()


# ── 動き方: 前進・左・右は MOVE だけ進み、左右は ROT だけ曲がりながら進む (MOVE_MODE=bc と同じ運動学)。
#    後退は向きを変えずに MOVE だけ下がる。建物のマスや地図の外へは進めず、その場に留まる (ぶつかる)。
@torch.no_grad()
def step_env(x, y, th, a, mid):
    th = th + torch.where(a == 1, -ROT, torch.where(a == 2, ROT, 0.0))
    sgn = torch.where(a == 3, -1.0, 1.0)
    nx, ny = x + sgn * MOVE * torch.cos(th), y + sgn * MOVE * torch.sin(th)
    inb = (nx > 0.05) & (nx < GRID - 0.05) & (ny > 0.05) & (ny < GRID - 0.05)
    free = BT[mid, nx.long().clamp(0, GRID - 1), ny.long().clamp(0, GRID - 1)] < 0
    ok = inb & free
    return torch.where(ok, nx, x), torch.where(ok, ny, y), th, ok


def random_starts(mid):
    """建物でないマスにランダムに置く"""
    xs, ys = [], []
    for m in mid.tolist():
        free = np.argwhere(MAPS[m] != BUILDING)
        r, c = free[np.random.randint(len(free))]
        xs.append(r + np.random.uniform(0.2, 0.8)); ys.append(c + np.random.uniform(0.2, 0.8))
    th = torch.rand(len(xs), device=DEVICE) * 2 * math.pi
    return torch.tensor(xs, device=DEVICE, dtype=torch.float32), torch.tensor(ys, device=DEVICE, dtype=torch.float32), th


class Explorer:
    """データ集め用の行動: 同じ行動をしばらく続け、ときどき変える (前進 4.5 割・左右 2 割ずつ・後退 1.5 割)"""
    def __init__(self, n):
        self.a = self.sample(n)
    def sample(self, n):
        return torch.multinomial(torch.tensor([0.45, 0.2, 0.2, 0.15], device=DEVICE), n, replacement=True)
    def __call__(self):
        change = torch.rand(len(self.a), device=DEVICE) < 0.25
        self.a = torch.where(change, self.sample(len(self.a)), self.a)
        return self.a


# 試しに 4 人分の目線を描く (道の上から、道の続く向きを見る)
def road_starts(mid):
    xs, ys, ths = [], [], []
    for m in mid.tolist():
        g = MAPS[m]; cand = np.argwhere(g == ROAD)
        r, c = cand[np.random.randint(len(cand))]
        dirs = [k for k, (dr, dc) in enumerate(((1, 0), (0, 1), (-1, 0), (0, -1)))
                if 0 <= r + dr < GRID and 0 <= c + dc < GRID and g[r + dr, c + dc] == ROAD] or [0]
        xs.append(r + 0.5); ys.append(c + 0.5); ths.append(np.random.choice(dirs) * math.pi / 2 + np.random.uniform(-0.3, 0.3))
    f = lambda v: torch.tensor(v, device=DEVICE, dtype=torch.float32)
    return f(xs), f(ys), f(ths)

mid = torch.tensor([0, 0, 1, N_MAPS_TRAIN], device=DEVICE)
x, y, th = road_starts(mid)
show_rows({"目線": list(render(x, y, th, mid).permute(0, 2, 3, 1).cpu().numpy())},
          [f"地図 {m}" for m in mid.tolist()], size=2.6)

In [ ]:
# ── データ集め: 歩かせながら毎歩「描画 → DINOv2 の特徴」を記録する ──
# 1 エピソード = EP_LEN+1 フレームと EP_LEN 個の行動。全フレームを 1 列に並べて持つ。
def collect(map_ids, ep_per_map, L, batch=128):
    mids = torch.tensor([m for m in map_ids for _ in range(ep_per_map)], device=DEVICE)
    E = len(mids); M = E * (L + 1)
    feats = torch.empty(M, N_TOK, FEAT_DIM, dtype=torch.float16, device=DEVICE)
    thumbs = torch.empty(M, THUMB, THUMB, 3, dtype=torch.uint8, device=DEVICE)
    acts = torch.zeros(M, dtype=torch.long, device=DEVICE)
    poses = torch.zeros(M, 3, device=DEVICE)                 # 評価用に本当の位置と向きも残す
    for b in range(0, E, batch):
        mid = mids[b:b + batch]; n = len(mid)
        x, y, th = random_starts(mid); pol = Explorer(n)
        rows = (torch.arange(b, b + n, device=DEVICE) * (L + 1))
        for t in range(L + 1):
            img = render(x, y, th, mid)
            feats[rows + t] = dino_encode(img, TOK_GRID).half()
            thumbs[rows + t] = (F.interpolate(img, size=THUMB, mode="bilinear", antialias=True).clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1)
            poses[rows + t] = torch.stack([x, y, th], 1)
            if t < L:
                a = pol(); acts[rows + t] = a
                x, y, th, _ = step_env(x, y, th, a, mid)
    ep_start = list(range(0, M, L + 1))
    return dict(feats=feats, thumbs=thumbs, acts=acts, poses=poses, mids=mids,
                ep_start=ep_start, ep_len=[L + 1] * E)

t0 = time.time()
train = collect(range(N_MAPS_TRAIN), EP_PER_MAP, EP_LEN)
test = collect(range(N_MAPS_TRAIN, N_MAPS_TRAIN + N_MAPS_TEST), 4, EP_LEN)
print(f"学習 {len(train['feats']):,} フレーム / 評価 {len(test['feats']):,} フレーム ({time.time()-t0:.0f}s)")

# 特徴を学習データの平均と標準偏差で標準化する (以後はすべてこの空間で考える)
FEAT_MU = train["feats"].float().mean((0, 1))
FEAT_SD = train["feats"].float().std((0, 1)) + 1e-4
for d in (train, test):
    d["feats"] = ((d["feats"].float() - FEAT_MU) / FEAT_SD).half()

def observe(x, y, th, mid):
    """本物の環境で目線を描いて、標準化した特徴にする"""
    return (dino_encode(render(x, y, th, mid), TOK_GRID) - FEAT_MU) / FEAT_SD

## 学習する

In [ ]:
wm = WorldModel(FEAT_DIM, N_TOK, HIST, n_actions=len(ACTIONS)).to(DEVICE)
print(f"世界モデル {sum(p.numel() for p in wm.parameters())/1e6:.1f}M パラメータ")
starts = valid_starts(train["ep_start"], train["ep_len"], HIST)
losses = train_wm(wm, train["feats"], train["acts"], starts, WM_STEPS)
plt.figure(figsize=(6, 3)); plt.plot(np.convolve(losses, np.ones(50) / 50, "valid")); plt.yscale("log")
plt.title("世界モデルの学習 (次の特徴の二乗誤差)"); plt.xlabel("step"); plt.show()

In [ ]:
dec = Decoder(FEAT_DIM, TOK_GRID).to(DEVICE)
train_decoder(dec, train["feats"], train["thumbs"], DEC_STEPS)
i = torch.arange(0, len(test["feats"]), max(1, len(test["feats"]) // 8))[:8]
show_rows({"実際": list(test["thumbs"][i].cpu().numpy() / 255), "特徴から復元": list(decode(dec, test["feats"][i]))},
          size=1.8, title="見たことのない街で、特徴 → 画像 の復元 (これが「想像」を見るときの画質の上限)")

## 1. どのくらい先まで当たるか

評価用の街 (学習で一度も見ていない) で、実際に見た直近 3 フレームと、その先の行動だけを渡して 15 歩先まで想像させ、
本物の特徴と比べます。比べる相手は「何も変わらないと予想する」(最後に見たフレームをそのまま使う) 素朴なやり方です。

In [ ]:
K = 15
def eval_windows(data, n_per_ep=4):
    """各エピソードから (文脈の先頭 i) を選ぶ。i..i+HIST-1 が文脈、その先 K 歩が答え"""
    out = []
    for s, n in zip(data["ep_start"], data["ep_len"]):
        hi = n - HIST - K
        if hi > 0:
            out += [s + int(o) for o in np.linspace(0, hi - 1, n_per_ep)]
    return torch.tensor(out, device=DEVICE)

w = eval_windows(test)
ctx = test["feats"][w[:, None] + torch.arange(HIST, device=DEVICE)].float()
acts_seq = test["acts"][w[:, None] + torch.arange(HIST - 1 + K, device=DEVICE)]
truth = test["feats"][w[:, None] + HIST + torch.arange(K, device=DEVICE)].float()
pred = imagine(wm, ctx, acts_seq)
err_wm = ((pred - truth) ** 2).mean((0, 2, 3)).cpu().numpy()
err_copy = ((ctx[:, -1:] - truth) ** 2).mean((0, 2, 3)).cpu().numpy()

plt.figure(figsize=(6, 3.2))
plt.plot(range(1, K + 1), err_copy, "o-", label="何も変わらないと予想")
plt.plot(range(1, K + 1), err_wm, "o-", label="世界モデルの想像")
plt.xlabel("何歩先か"); plt.ylabel("特徴の二乗誤差 (小さいほど良い)"); plt.legend(); plt.grid(alpha=.3)
plt.title("見たことのない街での予測"); plt.show()
print("1 歩先:  世界モデル %.3f / 何も変わらない %.3f" % (err_wm[0], err_copy[0]))
print("%d 歩先: 世界モデル %.3f / 何も変わらない %.3f" % (K, err_wm[-1], err_copy[-1]))

# 想像を画像にして見比べる
j = 0
steps = [0, 1, 3, 7, 11, 14]
real = test["thumbs"][w[j] + HIST + torch.tensor(steps, device=DEVICE)].cpu().numpy() / 255
show_rows({"実際": list(real), "想像": list(decode(dec, pred[j, steps]))},
          [f"{s+1} 歩先" for s in steps], size=1.8,
          title="行動: " + " ".join(ACTIONS[a] for a in acts_seq[j, HIST - 1:].tolist()))
display(animate_rows({"実際": test["thumbs"][w[j] + HIST + torch.arange(K, device=DEVICE)].cpu().numpy() / 255,
                      "想像": decode(dec, pred[j])}))

## 2. 「もし別の行動をしたら」を想像する

同じ場所から、行動の計画だけを変えて想像させます。本物の描画器でも同じ行動を実行して並べます。
建物に向かってまっすぐ進む計画では、ぶつかって景色が止まるところまで想像できているかが見どころです。

In [ ]:
PLANS = {"ずっと前進": [0] * 12, "ずっと左": [1] * 12, "ずっと右": [2] * 12, "左 6 → 右 6": [1] * 6 + [2] * 6,
         "前進 6 → 後退 6": [0] * 6 + [3] * 6}

def true_rollout(x, y, th, mid, plan):
    """本物の環境で実行して、各歩の特徴と縮小画像を返す"""
    fs, ims = [], []
    for a in plan:
        x, y, th, _ = step_env(x, y, th, torch.tensor([a], device=DEVICE), mid)
        img = render(x, y, th, mid)
        fs.append((dino_encode(img, TOK_GRID) - FEAT_MU) / FEAT_SD)
        ims.append(F.interpolate(img, size=THUMB, mode="bilinear", antialias=True)[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy())
    return torch.cat(fs), np.stack(ims)

# 評価用の街で、文脈の 3 フレームはまっすぐ前進して集める
torch.manual_seed(3); np.random.seed(3)
mid = torch.tensor([N_MAPS_TRAIN], device=DEVICE)
x, y, th = random_starts(mid)
ctx = [observe(x, y, th, mid)]
for _ in range(HIST - 1):
    x, y, th, _ = step_env(x, y, th, torch.tensor([0], device=DEVICE), mid)
    ctx.append(observe(x, y, th, mid))
ctx = torch.stack(ctx, 1)
steps = [0, 2, 5, 8, 11]
anim_rows = {}
for name, plan in PLANS.items():
    a = torch.tensor([[0] * (HIST - 1) + plan], device=DEVICE)
    imag = imagine(wm, ctx, a)[0]
    _, real = true_rollout(x, y, th, mid, plan)
    show_rows({"実際": list(real[steps]), "想像": list(decode(dec, imag[steps]))},
              [f"{s+1} 歩先" for s in steps], size=1.7, title=name)
    anim_rows[name + " 想像"] = decode(dec, imag)
display(animate_rows(anim_rows))

## 3. 映像から行動を当てる (逆動力学)

連続する 2 フレームの特徴だけから「その間に取った行動」を当てます。前回の説明の「映像 → 座標」にあたる部分です。
壁にぶつかって動けなかった前進は、2 枚の絵が同じになるので原理的に当てられません。

In [ ]:
inv = InverseDynamics(FEAT_DIM, N_TOK, len(ACTIONS)).to(DEVICE)
opt = torch.optim.AdamW(inv.parameters(), lr=1e-3, weight_decay=0.01)
pairs = valid_starts(train["ep_start"], train["ep_len"], 1).to(DEVICE)
for step in range(100 if QUICK else 3000):
    i = pairs[torch.randint(0, len(pairs), (256,), device=DEVICE)]
    loss = F.cross_entropy(inv(train["feats"][i].float(), train["feats"][i + 1].float()), train["acts"][i])
    opt.zero_grad(); loss.backward(); opt.step()
inv.eval()

tp = valid_starts(test["ep_start"], test["ep_len"], 1).to(DEVICE)
with torch.no_grad():
    guess = torch.cat([inv(test["feats"][c].float(), test["feats"][c + 1].float()).argmax(1) for c in tp.split(2048)])
truth_a = test["acts"][tp]
moved = (test["poses"][tp + 1, :2] - test["poses"][tp, :2]).norm(dim=1) > 1e-4
acc = (guess == truth_a).float().mean().item()
acc_moved = (guess == truth_a)[moved].float().mean().item()
major = torch.bincount(truth_a, minlength=NA).max().item() / len(truth_a)
print(f"正解率 {acc:.1%} (動けた歩だけなら {acc_moved:.1%}) / いつも一番多い行動と答えた場合 {major:.1%}")
cm = torch.zeros(NA, NA, dtype=torch.long)
for t, g in zip(truth_a.tolist(), guess.tolist()): cm[t, g] += 1
print("混同行列 (行 = 本当の行動, 列 = 当てた行動):", ACTIONS)
print(cm.numpy())

## 4. 想像の中で試して、ゴールへ向かう (計画)

ゴールの**画像だけ**を渡して、そこへ行かせます。地図も座標も渡しません。

1. ゴールは「スタートから適当に 10 歩歩いた場所の目線」です (なので必ずたどり着ける)
2. 毎歩、行動の候補 (10 歩分の並び) をたくさん想像の中で試し、ゴールの特徴に一番近づく並びを選ぶ (CEM)
3. その最初の 1 歩だけを**本物の環境で**実行して、新しく見えた絵からもう一度考え直す (MPC)

ゴールに 0.5 マス以内・向き 25° 以内まで近づけたら成功です。同じ歩数でランダムに歩いた場合と比べます。
毎歩、何千通りもの未来を想像するので、このノートブックでいちばん時間のかかるセルです。

In [ ]:
PLAN_T, POP, ITERS, ELITE = (4, 16, 2, 4) if QUICK else (10, 256, 4, 32)
GOAL_STEPS, MAX_STEPS = (3, 4) if QUICK else (10, 25)
N_TRIALS = 2 if QUICK else 20   # ここがいちばん時間のかかるセル。10 回ごとに途中経過を出す

@torch.no_grad()
def cem_plan(z_hist, a_hist, z_goal):
    """z_hist: (C,N,D) 直近の特徴 / a_hist: (C-1,) その間の行動 / z_goal: (N,D) → 一番よい行動の並び"""
    probs = torch.full((PLAN_T, NA), 1 / NA, device=DEVICE)
    for _ in range(ITERS):
        seq = torch.multinomial(probs, POP, replacement=True).T                  # (POP, PLAN_T)
        a = torch.cat([a_hist[None].expand(POP, -1), seq], 1)
        zf = imagine(wm, z_hist[None].expand(POP, -1, -1, -1), a)               # (POP, PLAN_T, N, D)
        cost = ((zf - z_goal) ** 2).mean((2, 3)).min(1).values                  # 途中でゴールに着けばよい
        elite = seq[cost.topk(ELITE, largest=False).indices]
        probs = 0.9 * F.one_hot(elite, NA).float().mean(0) + 0.1 / NA              # 少しだけ探索を残す
    return elite[0]

def angle_diff(a, b):
    return abs(math.atan2(math.sin(a - b), math.cos(a - b)))

def reached(p, g):
    return math.hypot(p[0] - g[0], p[1] - g[1]) < 0.5 and angle_diff(p[2], g[2]) < math.radians(25)

def trial(seed, use_model=True):
    torch.manual_seed(seed); np.random.seed(seed)
    mid = torch.tensor([N_MAPS_TRAIN + seed % N_MAPS_TEST], device=DEVICE)
    x, y, th = random_starts(mid)
    # ゴール: スタートから 10 歩歩いた先 (十分に離れた場所になるまで引き直す)
    for _ in range(20):
        gx, gy, gth = x, y, th; pol = Explorer(1)
        for _ in range(GOAL_STEPS):
            gx, gy, gth, _ = step_env(gx, gy, gth, pol(), mid)
        g = (gx.item(), gy.item(), gth.item())
        if not reached((x.item(), y.item(), th.item()), g): break
    z_goal = observe(gx, gy, gth, mid)[0]
    zs, acts, path = [observe(x, y, th, mid)[0]], [], [(x.item(), y.item())]
    for step in range(MAX_STEPS):
        if reached((x.item(), y.item(), th.item()), g):
            return True, step, path, g, mid.item()
        if use_model:
            C = min(len(zs), HIST)                      # 直近 C フレームと、その間の C-1 個の行動
            a_hist = torch.tensor(acts[len(acts) - (C - 1):], dtype=torch.long, device=DEVICE)
            a = cem_plan(torch.stack(zs[-C:]), a_hist, z_goal)[0]
        else:
            a = torch.randint(0, NA, (), device=DEVICE)
        x, y, th, _ = step_env(x, y, th, a.view(1), mid)
        zs.append(observe(x, y, th, mid)[0]); acts.append(a.item()); path.append((x.item(), y.item()))
    return reached((x.item(), y.item(), th.item()), g), MAX_STEPS, path, g, mid.item()

t0 = time.time()
res_wm, res_rand = [], []
for s in range(N_TRIALS):
    res_wm.append(trial(s, True)); res_rand.append(trial(s, False))
    if (s + 1) % 10 == 0:
        print(f"  {s+1}/{N_TRIALS} 回  計画 {np.mean([r[0] for r in res_wm]):.0%} / ランダム {np.mean([r[0] for r in res_rand]):.0%}  ({time.time()-t0:.0f}s)")
ok_wm = np.mean([r[0] for r in res_wm]); ok_rand = np.mean([r[0] for r in res_rand])
print(f"成功率: 世界モデルで計画 {ok_wm:.0%} / ランダムに歩く {ok_rand:.0%}  ({N_TRIALS} 回ずつ, {time.time()-t0:.0f}s)")
steps_ok = [r[1] for r in res_wm if r[0]]
if steps_ok: print(f"成功したときの歩数: 平均 {np.mean(steps_ok):.1f} 歩 (ゴールはスタートから {GOAL_STEPS} 歩の場所)")

# 道のりを地図に描く (青 = 計画, 灰 = ランダム, ★ = ゴール)
fig, ax = plt.subplots(1, 4, figsize=(13, 3.4))
for a, k in zip(ax, range(min(4, N_TRIALS))):
    ok, n, path, g, m = res_wm[k]; _, _, rpath, _, _ = res_rand[k]
    p = np.array(path); rp = np.array(rpath)
    lo = np.floor(np.minimum(p.min(0), g[:2]) - 2).clip(0); hi = np.ceil(np.maximum(p.max(0), g[:2]) + 2).clip(max=GRID)
    a.imshow(MAPS[m].T, cmap=cmap, vmin=0, vmax=3, origin="lower", extent=(0, GRID, 0, GRID))
    a.plot(rp[:, 0], rp[:, 1], "-", color="#888", lw=1.5)
    a.plot(p[:, 0], p[:, 1], "-", color="#2a7fff", lw=2); a.plot(*p[0], "o", color="#2a7fff")
    a.plot(g[0], g[1], "*", color="gold", ms=14, mec="k")
    a.set_xlim(lo[0], hi[0]); a.set_ylim(lo[1], hi[1]); a.set_xticks([]); a.set_yticks([])
    a.set_title(("成功" if ok else "失敗") + f" {n} 歩", fontsize=10)
plt.show()

## 5. 学習したモデルを保存して Hugging Face に置く

ここまでの `wm` (世界モデル)・`dec` (復元器)・`inv` (逆動力学) は、**Colab のメモリの中にしかありません**。ランタイムを切ると消えます。
このセルで `mesa_wm_export/` に書き出し、`HF_MODEL_REPO` を書けば Hugging Face のモデルリポジトリにも上げます。

| ファイル | 中身 |
| --- | --- |
| `world_model.safetensors` | 世界モデル (順動力学) の重み |
| `decoder.safetensors` | 特徴 → 64×64 の画像の復元器 |
| `inverse_dynamics.safetensors` | 逆動力学 |
| `feat_stats.safetensors` | DINOv2 の特徴を標準化する平均と標準偏差 (これが無いと世界モデルに入れられない) |
| `config.json` | 大きさ・行動・描画の設定。シミュレーター (Space) はここを読んで同じ条件で描く |

上げるには Hugging Face の**書き込み用トークン**が要ります。Colab の左の鍵のマーク (シークレット) に `HF_TOKEN` として入れておくと、
`login()` がそれを使います。

In [ ]:
EXPORT_DIR = "mesa_wm_export"
HF_MODEL_REPO = ""    # 例: "あなたのユーザー名/mesa-world-model"。空のままなら書き出すだけ

from safetensors.torch import save_file
os.makedirs(EXPORT_DIR, exist_ok=True)
def save_sd(model, name):
    save_file({k: v.detach().float().cpu().contiguous() for k, v in model.state_dict().items()}, f"{EXPORT_DIR}/{name}")
save_sd(wm, "world_model.safetensors")
save_sd(dec, "decoder.safetensors")
save_sd(inv, "inverse_dynamics.safetensors")
save_file({"mu": FEAT_MU.float().cpu().contiguous(), "sd": FEAT_SD.float().cpu().contiguous()}, f"{EXPORT_DIR}/feat_stats.safetensors")

config = dict(
    format_version=1,
    actions=ACTIONS,                       # 行動の番号 = この並び
    feat_dim=FEAT_DIM, tok_grid=TOK_GRID, hist=HIST,
    world_model=dict(d=256, depth=6, heads=8), decoder=dict(ch=128), inverse_dynamics=dict(r=32),
    img=IMG, thumb=THUMB, grid=GRID, move=MOVE, rot_deg=math.degrees(ROT),
    render=dict(fov_deg=math.degrees(FOV), eye=EYE, ssaa=SSAA, layers=LAYERS, tlayers=TLAYERS,
                tw=TW, th=TH, gt=GT, at=AT, tree_h=TREE_H, tree_r=TREE_R, fog=[FOG0, FOG1],
                textures=RAW, p_big=P_BIG),
    maps=dict(train_seeds=MAP_SEEDS[:N_MAPS_TRAIN], test_seeds=MAP_SEEDS[N_MAPS_TRAIN:]),
    dino="facebookresearch/dinov2:dinov2_vits14",
    train=dict(wm_steps=WM_STEPS, dec_steps=DEC_STEPS, ep_per_map=EP_PER_MAP, ep_len=EP_LEN, quick=QUICK),
)
with open(f"{EXPORT_DIR}/config.json", "w") as f:
    json.dump(config, f, ensure_ascii=False, indent=1)
print("書き出し:", sorted(os.listdir(EXPORT_DIR)))

if HF_MODEL_REPO:
    from huggingface_hub import HfApi, login
    login()                                # シークレットの HF_TOKEN か、入力欄のトークンを使う
    api = HfApi()
    api.create_repo(HF_MODEL_REPO, exist_ok=True)
    api.upload_folder(folder_path=EXPORT_DIR, repo_id=HF_MODEL_REPO, commit_message="world model weights")
    print(f"アップロードしました: https://huggingface.co/{HF_MODEL_REPO}")
else:
    print("HF_MODEL_REPO が空なので、アップロードはしていません (左のファイル欄からダウンロードもできます)")

## うまくいかないときに触るところ

- **想像がすぐぼやける**: `WM_STEPS` を増やす、`EP_PER_MAP` を増やしてデータを足す
- **計画の成功率が低い**: `POP` と `ITERS` を増やす。ゴールの特徴に近づくのに位置が合わないときは、
  見た目が似た場所を取り違えています (特徴だけで考える方式の限界。過去のフレームも手がかりに使うと改善します)
- **もっと細かく見たい**: `TOK_GRID = 16` (トークンが 4 倍になり、学習も 4 倍ほど遅くなる)

## 次の一歩

- 学習した `wm` を住民の頭に入れる: 「目的の店の外観 (画像) をゴールにして、地図なしで歩く」が、そのまま試せる形になっています
- 行動にスキル (walk / carry / stroll) や速さを足すと、BC の行動空間と揃います
- 書き出したモデルを Hugging Face の Space (Gradio) に載せると、前進・左・右・後退のボタンで「想像の中を歩く」シミュレーターになります (`hf/` フォルダ)